In [ ]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import numpy as np
import re
import math
import tldextract
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import joblib
import os

# ---------------- Feature Extraction Functions ---------------- #

SHORTENERS = [
    "bit.ly", "goo.gl", "t.co", "tinyurl.com", "ow.ly",
    "is.gd", "buff.ly", "adf.ly", "shorturl.at"
]

def calculate_entropy(s):
    prob = [float(s.count(c)) / len(s) for c in dict.fromkeys(s)]
    return round(-sum([p * math.log(p) / math.log(2.0) for p in prob]), 4)

def get_domain_age(domain):
    # No whois lookup; set domain age to 0
    return 0.0

def check_blacklist_mock(url):
    keywords = ["secure", "login", "verify", "bank", "kyc", "claim", "refund"]
    return int(any(k in url.lower() for k in keywords))

def extract_features_from_input(url, source_channel, sender_known, clicked):
    parsed = tldextract.extract(url)
    domain = parsed.domain + "." + parsed.suffix
    return {
        "url_length": len(url),
        "has_https": int(url.lower().startswith("https")),
        "is_shortened": int(domain in SHORTENERS),
        "entropy_score": calculate_entropy(domain),
        "domain_age_months": get_domain_age(domain),
        "blacklist_match": check_blacklist_mock(url),
        "source_channel": source_channel,
        "sender_known": int(sender_known),
        "clicked": int(clicked)
    }

# ---------------- Train Model If Not Exists ---------------- #

MODEL_FILENAME = "url_fraud_rf_gui_model.pkl"

def train_model():
    try:
        df = pd.read_csv("url_fraud_dataset.csv")  # Must exist

        feature_df = df["url"].apply(
            lambda url: extract_features_from_input(url, "SMS", 0, 0)
        ).apply(pd.Series)

        feature_df["source_channel"] = df["source_channel"]
        feature_df["sender_known"] = df["sender_known"]
        feature_df["clicked"] = df["clicked"]
        feature_df["fraud_label"] = df["fraud_label"]

        X = feature_df.drop("fraud_label", axis=1)
        y = feature_df["fraud_label"]

        preprocessor = ColumnTransformer(
            transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), ["source_channel"])],
            remainder='passthrough'
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(n_estimators=150, random_state=42))
        ])

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, stratify=y, test_size=0.2, random_state=42
        )

        pipeline.fit(X_train, y_train)
        joblib.dump(pipeline, MODEL_FILENAME)

        print("✅ Model trained and saved!")

        # ---------------- Evaluation Metrics ---------------- #
        y_pred = pipeline.predict(X_test)

        print("\n===== MODEL EVALUATION =====")
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
        print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
        print("Precision:", round(precision_score(y_test, y_pred), 4))
        print("Recall:", round(recall_score(y_test, y_pred), 4))
        print("F1 Score:", round(f1_score(y_test, y_pred), 4))
        print("\nClassification Report:\n", classification_report(y_test, y_pred))

        return pipeline

    except Exception as e:
        print("❌ Error training model:", e)
        return None

# ---------------- Load or Train Model ---------------- #

if os.path.exists(MODEL_FILENAME):
    model = joblib.load(MODEL_FILENAME)
    print("✅ Model loaded from disk.")
else:
    model = train_model()

if model is None:
    print("⚠️ Exiting: Model could not be loaded or trained.")
    exit()

# ---------------- GUI ---------------- #

root = tk.Tk()
root.title("🔐 URL Fraud Detector")
root.geometry("500x350")

tk.Label(root, text="Paste URL:").pack(pady=5)
url_entry = tk.Entry(root, width=60)
url_entry.pack()

tk.Label(root, text="Source (SMS/Email/WhatsApp):").pack(pady=5)
source_var = tk.StringVar(value="SMS")
source_entry = tk.OptionMenu(root, source_var, "SMS", "Email", "WhatsApp", "Other")
source_entry.pack()

sender_var = tk.IntVar()
clicked_var = tk.IntVar()

tk.Checkbutton(root, text="Sender is Known", variable=sender_var).pack()
tk.Checkbutton(root, text="User Clicked the Link", variable=clicked_var).pack()

def predict():
    url = url_entry.get().strip()
    if not url:
        messagebox.showwarning("Input Missing", "Please paste a URL.")
        return
    try:
        features = extract_features_from_input(
            url,
            source_var.get(),
            sender_var.get(),
            clicked_var.get()
        )
        input_df = pd.DataFrame([features])
        result = model.predict(input_df)[0]
        prob = model.predict_proba(input_df)[0][1]

        if result == 1:
            messagebox.showerror(
                "🚨 FRAUD DETECTED",
                f"URL is likely FRAUDULENT\nRisk Score: {round(prob*100)}%"
            )
        else:
            messagebox.showinfo(
                "✅ Safe URL",
                f"URL appears safe\nRisk Score: {round(prob*100)}%"
            )

    except Exception as e:
        messagebox.showerror("Error", str(e))

tk.Button(
    root,
    text="Check URL",
    command=predict,
    bg="blue",
    fg="white",
    font=("Arial", 12)
).pack(pady=20)

root.mainloop()

✅ Model trained and saved!

===== MODEL EVALUATION =====
Confusion Matrix:
 [[135   4]
 [  3  58]]
Accuracy: 0.965
Precision: 0.9355
Recall: 0.9508
F1 Score: 0.9431

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.97       139
           1       0.94      0.95      0.94        61

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.97      0.96      0.97       200

